# 📘 Notebook 17: Machine Learning with Spark MLlib

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 5 of 6 in this module · (17 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Turn a warehouse into a **training table**: one row per customer, a label, and features that do not leak the answer
- Build an MLlib **pipeline** of transformers and an estimator, and say why pipelines matter on a cluster
- Evaluate a classifier honestly: a **baseline**, a held-out **test set**, the **confusion matrix** and the **AUC**
- Compare two models, and read which features a tree relies on
- Segment customers with **k-means** and choose the number of clusters with the **silhouette**

> **Prerequisites:** Notebook 16 (this data in Spark). The ideas of machine learning (train and test sets, overfitting, logistic regression, decision trees, k-means, evaluation metrics) are taught in *From Python to Deep Learning and Fine-Tuning*, Notebooks 08 to 11. This notebook does not re-teach them: it shows how they run when the data is in Spark.
>
> 🔭 **Why this notebook matters:** a model trained on a laptop with scikit-learn needs all the training data in one machine's memory. When the data lives in a cluster, the training must go to the data. Spark's machine learning library, **MLlib**, provides the familiar algorithms in a form that runs on every machine of the cluster, organised as **pipelines**: the same preparation steps, fitted once on the training data and applied identically to any new data. The modelling questions stay the same; what changes is where the work happens and how carefully the steps are chained.

> 📦 **Libraries used in this notebook.** **PySpark** with its **MLlib** module, in local mode, and **pandas** and **Matplotlib** for small results. The cell below installs PySpark; in Google Colab it is already present.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

In [ ]:
%pip install -q pyspark

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

In [ ]:
import sys

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
spark = (SparkSession.builder.master("local[*]").appName("adventureworks-ml")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## 1. From a warehouse to a training table

### Intuition first
A model learns from **one row per thing we want to predict about**. Our question is the one the company's marketing team would ask: *which customers buy a bicycle?* A customer who is likely to buy one but has not yet is worth a brochure. So the unit is the **customer**, the **label** is 1 if they ever bought a bicycle and 0 otherwise, and the **features** are what the company knows about a person before they buy: age, income, family, education, occupation, home, cars, commute and country.

Notice what is *not* among the features: how much the customer spent, or how many orders they placed. Those are partly caused by buying a bicycle. A model given them would look brilliant in testing and be useless in practice, because for a new prospect they do not exist yet. Using information that would not be available at prediction time is called **leakage**, and it is the most common way that machine learning projects fool themselves.

In [ ]:
def read(name):
    return spark.read.csv(f"{DATA}/{name}.csv", header=True, inferSchema=True)

sales, product = read("FactInternetSales"), read("DimProduct")
subcategory, category = read("DimProductSubcategory"), read("DimProductCategory")
customer, geography = read("DimCustomer"), read("DimGeography")
last_day = sales.agg(F.max(F.to_date(F.substring("OrderDate", 1, 10)))).first()[0]

bike_keys = (product.join(subcategory, "ProductSubcategoryKey").join(category, "ProductCategoryKey")
                    .filter(F.col("EnglishProductCategoryName") == "Bikes").select("ProductKey"))
buyers = (sales.join(bike_keys, "ProductKey").select("CustomerKey").distinct()
               .withColumn("label", F.lit(1.0)))

data = (customer.join(geography.select("GeographyKey", "EnglishCountryRegionName"), "GeographyKey")
        .join(buyers, "CustomerKey", "left")
        .select("CustomerKey",
                F.coalesce("label", F.lit(0.0)).alias("label"),
                F.floor(F.months_between(F.lit(last_day), F.to_date("BirthDate")) / 12).alias("age"),
                F.col("YearlyIncome").cast("double").alias("income"),
                "TotalChildren", "NumberChildrenAtHome", "HouseOwnerFlag", "NumberCarsOwned",
                "Gender", "MaritalStatus", "EnglishEducation", "EnglishOccupation", "CommuteDistance",
                F.col("EnglishCountryRegionName").alias("country")))
data.cache()
data.show(3)

## 2. The baseline: how good is guessing?

Before any model, find out how well a model that ignores every feature would do. If a classifier cannot beat that, it has learned nothing.

In [ ]:
balance = data.groupBy("label").count().toPandas().set_index("label")["count"]
print(balance)
print(f"Always guessing the majority class is right {balance.max() / balance.sum():.1%} of the time.")

The classes are close to balanced, so accuracy is a fair first measure here. (When one class is rare, say 2 percent fraud, a model that always answers *no fraud* is 98 percent accurate and worthless; then precision, recall and the AUC are the measures to watch.)

## 3. A held-out test set

The test set must be put aside **before** anything is fitted, including the scaling of the features. `randomSplit` with a fixed seed makes the split reproducible.

In [ ]:
train, test = data.randomSplit([0.8, 0.2], seed=42)
print("training rows:", train.count(), "  test rows:", test.count())

## 4. An MLlib pipeline

### Intuition first
Text columns such as `EnglishEducation` must become numbers, the numbers must be gathered into one **feature vector**, and the vector must be scaled. Each step learns something from the data (the list of categories, the means and standard deviations), so each must be learned from the **training** data only and then applied unchanged to the test data and to any future customer. A **pipeline** packages those steps with the model, so that this happens automatically and can never be done in the wrong order. On a cluster the pipeline is also what gets shipped to the machines that apply it.

MLlib calls a step that learns from data an **estimator** (it has `fit`), and a step that only transforms a **transformer** (it has `transform`). Fitting a pipeline returns a **pipeline model**, which is a chain of transformers.

In [ ]:
categorical = ["Gender", "MaritalStatus", "EnglishEducation", "EnglishOccupation", "CommuteDistance", "country"]
numeric = ["age", "income", "TotalChildren", "NumberChildrenAtHome", "HouseOwnerFlag", "NumberCarsOwned"]

indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in categorical]
encoder = OneHotEncoder(inputCols=[c + "_idx" for c in categorical], outputCols=[c + "_vec" for c in categorical])
assembler = VectorAssembler(inputCols=numeric + [c + "_vec" for c in categorical], outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features")
prepare = indexers + [encoder, assembler, scaler]

logistic = Pipeline(stages=prepare + [LogisticRegression(maxIter=50)]).fit(train)
predictions = logistic.transform(test)
predictions.select("CustomerKey", "label", "probability", "prediction").show(5, truncate=False)

## 5. Evaluating honestly

Three views of the same predictions: **accuracy**, the **confusion matrix** (which mistakes), and the **AUC**, the probability that the model ranks a random buyer above a random non-buyer (0.5 is guessing, 1.0 is perfect). The AUC does not depend on the 0.5 threshold, which makes it the fairest single number for comparing models.

In [ ]:
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

def report(name, predictions):
    accuracy = predictions.select(F.avg((F.col("label") == F.col("prediction")).cast("double"))).first()[0]
    print(f"{name:<22} accuracy {accuracy:.3f}   AUC {auc.evaluate(predictions):.3f}")

report("logistic regression", predictions)
(predictions.groupBy("label").pivot("prediction").count()
            .withColumnRenamed("0.0", "predicted 0").withColumnRenamed("1.0", "predicted 1")
            .orderBy("label").show())

The model beats the baseline clearly, but it is far from perfect: demographics alone carry only part of the answer, which is what the weak correlations of Notebook 16 warned us about. That is a realistic result, and reporting it honestly is part of the work.

## 6. A second model, and what it relies on

A **decision tree** can capture interactions that logistic regression cannot (for example, *young and two cars*). It also tells us which features its splits use most. Only the last stage of the pipeline changes.

In [ ]:
tree = Pipeline(stages=prepare + [DecisionTreeClassifier(maxDepth=6, seed=42)]).fit(train)
tree_predictions = tree.transform(test)
report("logistic regression", predictions)
report("decision tree (depth 6)", tree_predictions)

The two measures disagree. The tree is right more often at the 0.5 threshold, yet it **ranks** customers worse. A tree of depth 6 has at most 64 leaves, so it gives only a few dozen distinct probabilities, and many customers tie; logistic regression gives every customer a different, smoothly varying score. If the task is *mail the 1,000 most likely buyers*, ranking is what matters and the AUC is the measure to trust. If it is *decide yes or no for each customer*, accuracy at a chosen threshold matters. Decide which question you are answering before you choose the winner.

Now, which features does the tree use?

In [ ]:
attrs = tree_predictions.schema["raw_features"].metadata["ml_attr"]["attrs"]   # names set by the assembler
names = {a["idx"]: a["name"] for group in attrs.values() for a in group}
importance = tree.stages[-1].featureImportances
top = sorted(((importance[i], names[i]) for i in range(len(names)) if importance[i] > 0), reverse=True)[:8]
for value, name in top:
    print(f"{name:<40} {value:.3f}")

The tree leans on a handful of features: income and the number of cars above all, then the number of children, age, whether the customer lives in Australia, and the commute. That ranking describes **this tree** and not the world: a different depth or seed changes it, and an importance is not a cause. Treat it as a reason to look further, for example by comparing the bike-buying rate across numbers of cars, not as a conclusion.

## 7. Without a label: segmenting customers

### Intuition first
Marketing also asks a question without a right answer: *what kinds of customers do we have?* **k-means** (Notebook 11 of the Deep Learning course) groups similar customers. We describe each customer by three classic behavioural numbers, known as **RFM**: **R**ecency (days since the last order), **F**requency (number of orders) and **M**onetary value (total spend), the last one on a log scale because it is so skewed (Notebook 16). Behaviour may be used here, because we are describing customers, not predicting a future label.

In [ ]:
rfm = (sales.groupBy("CustomerKey")
            .agg(F.datediff(F.lit(last_day), F.max(F.to_date(F.substring("OrderDate", 1, 10)))).alias("recency"),
                 F.countDistinct("SalesOrderNumber").alias("frequency"),
                 F.log10(F.sum("SalesAmount")).alias("monetary_log")))
rfm_vec = Pipeline(stages=[VectorAssembler(inputCols=["recency", "frequency", "monetary_log"], outputCol="raw"),
                           StandardScaler(inputCol="raw", outputCol="features", withMean=True)]).fit(rfm).transform(rfm)
rfm_vec.cache()

silhouette = ClusteringEvaluator()
scores = {}
for k in range(2, 8):
    model = KMeans(k=k, seed=42).fit(rfm_vec)
    scores[k] = silhouette.evaluate(model.transform(rfm_vec))
    print(f"k = {k}: silhouette {scores[k]:.3f}")

The **silhouette** measures how much closer each customer is to its own cluster than to the next one (from -1 to 1, higher is better). Choose a k with a high score that the business can also use; four or five segments can be named and acted on, eleven cannot. Let us describe the segments for the best-scoring k.

In [ ]:
best_k = max(scores, key=scores.get)
segments = KMeans(k=best_k, seed=42).fit(rfm_vec).transform(rfm_vec)
(segments.groupBy("prediction")
         .agg(F.count("*").alias("customers"), F.round(F.avg("recency")).alias("avg_recency_days"),
              F.round(F.avg("frequency"), 2).alias("avg_orders"),
              F.round(F.avg(F.pow(10, "monetary_log"))).alias("avg_spend"))
         .orderBy(F.desc("avg_spend")).show())

Each row is a segment that can be given a name, for example *recent big spenders*, *one-off accessory buyers*, *lapsed customers*, and each name suggests a different action. The clustering found the groups; naming them and deciding what to do is the analyst's job.

## 8. Where we stand

| Step | What MLlib provides | The trap it guards against |
|---|---|---|
| Training table | DataFrame joins and aggregations | leakage: features unknown at prediction time |
| Baseline | a one-line aggregation | celebrating a model that beats nothing |
| Preparation | indexers, encoders, assembler, scaler | fitting preparation steps on the test data |
| Model | a pipeline with an estimator at the end | steps applied in different orders in training and use |
| Evaluation | evaluators, confusion matrix by `groupBy` | trusting accuracy alone |
| Segmentation | k-means with the silhouette | choosing k by habit, or clusters nobody can use |

The models were trained on one machine, but nothing in the code would change on a cluster except the `.master(...)` line. The last notebook asks the question this course has been asking all along, now about this data: did we need a cluster at all?

---
## ✏️ Exercises

### Exercise 1 (A random forest)
Replace the last stage with a `RandomForestClassifier(numTrees=50, maxDepth=8, seed=42)` and compare its accuracy and AUC with the two models above.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from pyspark.ml.classification import RandomForestClassifier
forest = Pipeline(stages=prepare + [RandomForestClassifier(numTrees=50, maxDepth=8, seed=42)]).fit(train)
report("random forest", forest.transform(test))
report("decision tree (depth 6)", tree_predictions)
report("logistic regression", predictions)
```

*A forest averages many trees grown on different samples, so it usually beats a single tree. Whether the gain is worth fifty times the training work is a judgement to make, and to report.*

</details>

### Exercise 2 (Tuning with cross-validation)
Use `CrossValidator` with 3 folds and a `ParamGridBuilder` over `regParam` in `[0.0, 0.01, 0.1]` for the logistic regression, choosing by AUC. Report the best `regParam` and the test AUC of the best model.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
lr = LogisticRegression(maxIter=50)
grid = ParamGridBuilder().addGrid(lr.regParam, [0.0, 0.01, 0.1]).build()
cv = CrossValidator(estimator=Pipeline(stages=prepare + [lr]), estimatorParamMaps=grid,
                    evaluator=auc, numFolds=3, seed=42).fit(train)
print("mean AUC per setting:", [round(m, 4) for m in cv.avgMetrics])
print("best regParam:", cv.bestModel.stages[-1].getRegParam())
report("tuned logistic", cv.bestModel.transform(test))
```

*Cross-validation uses only the training data, so the test set stays untouched until the end. Notice that the whole pipeline is cross-validated, preparation included: that is what keeps each fold honest.*

</details>

### Exercise 3 (Moving the threshold)
A brochure costs money, so marketing wants fewer wasted mailings. Using the logistic regression's probabilities, compute **precision** and **recall** for the thresholds 0.3, 0.5 and 0.7. (Use `vector_to_array` from `pyspark.ml.functions` to read the probability of class 1.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from pyspark.ml.functions import vector_to_array
p1 = predictions.withColumn("p1", vector_to_array("probability")[1])
for t in (0.3, 0.5, 0.7):
    c = p1.select(F.sum(((F.col("p1") >= t) & (F.col("label") == 1)).cast("int")).alias("tp"),
                  F.sum(((F.col("p1") >= t) & (F.col("label") == 0)).cast("int")).alias("fp"),
                  F.sum(((F.col("p1") < t) & (F.col("label") == 1)).cast("int")).alias("fn")).first()
    print(f"threshold {t}: precision {c.tp / (c.tp + c.fp):.3f}   recall {c.tp / (c.tp + c.fn):.3f}")
```

*A higher threshold mails fewer people, with a higher share of real buyers among them (precision up) and more buyers missed (recall down). The right threshold is a business decision about the cost of each kind of mistake.*

</details>

### Exercise 4 (Is it leakage? A little harder)
Add the customer's **number of orders** to the features (join it from `sales`), retrain the logistic regression and report the AUC. Then explain in two sentences why the improvement must not be reported as a better model for finding new bike buyers.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
orders = sales.groupBy("CustomerKey").agg(F.countDistinct("SalesOrderNumber").cast("double").alias("orders"))
train_l, test_l = [d.join(orders, "CustomerKey") for d in (train, test)]
leaky_assembler = VectorAssembler(inputCols=numeric + ["orders"] + [c + "_vec" for c in categorical],
                                  outputCol="raw_features")
leaky = Pipeline(stages=indexers + [encoder, leaky_assembler, scaler, LogisticRegression(maxIter=50)]).fit(train_l)
report("with number of orders", leaky.transform(test_l))
report("demographics only", predictions)
```

*The number of orders is known only after the customer has bought, and buying a bicycle is part of it, so the feature carries the answer into the question. For a new prospect it is zero or unknown, and the impressive score would vanish the day the model was used.*

</details>

## 🔑 Key takeaways

- Build the **training table** at the unit you predict about, and exclude anything not known at prediction time: that is **leakage**.
- Always compare with a **baseline**; with rare classes, prefer precision, recall and the **AUC** to accuracy.
- An MLlib **pipeline** fits every preparation step on the training data only and applies it identically everywhere.
- Feature importances describe a model, not the world; they are leads, not causes.
- **k-means** on RFM features gives segments; choose k with the **silhouette** and with what the business can use.
- The same code runs on a cluster: only the session's master changes.

**Next:** *Notebook 18: Measuring Scale*, the last notebook, where pandas, SQLite and Spark answer the same question on growing copies of this data, and we find where each one stops being the right tool.

In [ ]:
spark.stop()

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*